In [1]:
import sys
print(sys.executable)

/Users/Owner/Projects/lumina_cosmic_variance/.venv/bin/python


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
import h5py
from pathlib import Path
from scipy.integrate import quad

In [3]:
#Lumina planck cosmology parameters (methods paper Table 2)
lumina_cosmology=FlatLambdaCDM(
    H0=67.66*u.km/u.s/u.Mpc,
    Om0=0.3096,
    Ob0=4.897/100,
    Tcmb0=2.7255*u.K,
    Neff=3.046,
    m_nu=0.0*u.eV,
)

#experimental design
survey_area=9.7*u.arcmin**2
delta_z=1.0
mass_bin_width=0.5

lumina_cosmology

FlatLambdaCDM(name=None, H0=<Quantity 67.66 km / (Mpc s)>, Om0=0.3096, Tcmb0=<Quantity 2.7255 K>, Neff=3.046, m_nu=<Quantity [0., 0., 0.] eV>, Ob0=0.04897)

In [ ]:
hdf5_path=Path("placeholder")

if not hdf5_path.is_file():
    raise FileNotFoundError(f"HDF5 file not found at {hdf5_path}")

with h5py.File(hdf5_path, "r") as f:
    print(list(f.keys()))

In [10]:
#find volume integration
def calculate_field_volume(cosmology, z_center, delta_z, survey_area):
    z_min=z_center-delta_z/2
    z_max=z_center+delta_z/2

    def differential_volume(z):
        return cosmology.differential_comoving_volume(z).value
    
    volume,error= quad(differential_volume,z_min,z_max) #numerical values
    field_solid_angle=survey_area.to(u.sr)

    field_volume=volume*u.Mpc**3/u.sr*field_solid_angle #Mpc^3 units
    field_volume_error=error*u.Mpc**3/u.sr*field_solid_angle #Mpc^3 units

    los_depth=cosmology.comoving_distance(z_max)-cosmology.comoving_distance(z_min)

    return field_volume,field_volume_error,los_depth

test_z=11.931
field_volume,field_volume_error,los_depth=calculate_field_volume(
    cosmology=lumina_cosmology,
    z_center=test_z,
    delta_z=delta_z,
    survey_area=survey_area,
)

print("field volume:", field_volume)
print("field volume error",field_volume_error)
print("los depth", los_depth)

field volume: 14096.932083667518 Mpc3
field volume error 1.565073857586476e-10 Mpc3
los depth 170.99971462774192 Mpc


In [15]:
#size of each box

#rectangle (pencil beam)
cross_sectional_area=field_volume/los_depth
length=np.sqrt(cross_sectional_area)
rectangle_dim=(length,length,los_depth)

#cube
cube_length=np.cbrt(field_volume)
cube_dim=(cube_length,cube_length,cube_length)

print("field volume:", field_volume)
print("rectangle (pencil beam) dim:", rectangle_dim)
print("cube dim:", cube_dim)

field volume: 14096.932083667518 Mpc3
rectangle (pencil beam) dim: (<Quantity 9.07955599 Mpc>, <Quantity 9.07955599 Mpc>, <Quantity 170.99971463 Mpc>)
cube dim: (<Quantity 24.15691859 Mpc>, <Quantity 24.15691859 Mpc>, <Quantity 24.15691859 Mpc>)
